# Notebook 07 — Dengue Anomaly Detection and Outbreak Identification

**RiskWeave | Sri Lanka**

## Objective
Detect unusually high district-level dengue observations using historical dengue counts.

**Scope:** This first version uses dengue counts only. Weather anomaly detection is not included yet.

**Important:** A statistical anomaly is not automatically a confirmed dengue outbreak. Until validated against official epidemiological evidence and outbreak definitions, outputs are labelled *candidate high anomalies*.

### Method
1. Validate the cleaned district-week dataset.
2. Inspect reporting-period lengths.
3. Convert counts to a seven-day-equivalent rate to account for 6-, 7-, and 8-day reporting periods.
4. Calculate a district-specific baseline from the preceding 52 reporting periods, excluding the current period.
5. Calculate a robust score using the rolling median and median absolute deviation (MAD).
6. Flag candidate high anomalies using a provisional threshold of 3.5.
7. Review and export results.

The threshold is an exploratory statistical screening heuristic, not an official Sri Lankan outbreak threshold.

## 1. Import libraries and locate the RiskWeave project

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)

cwd = Path.cwd()
if cwd.name.lower() == "notebooks":
    PROJECT_ROOT = cwd.parent
elif (cwd / "data").exists():
    PROJECT_ROOT = cwd
else:
    raise FileNotFoundError(
        "Open this notebook from RiskWeave/notebooks or the project root."
    )

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "dengue_weather_weekly_25_districts.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "anomaly_detection"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Input:", DATA_PATH)
print("Output:", OUTPUT_DIR)

## 2. Load the processed dataset

This dataset contains aligned district-week dengue counts and weather summaries. This notebook uses only the dengue count and reporting dates.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)

required_cols = ["district_key", "week_start", "week_end", "dengue_cases"]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df["week_start"] = pd.to_datetime(df["week_start"], errors="coerce")
df["week_end"] = pd.to_datetime(df["week_end"], errors="coerce")
df["dengue_cases"] = pd.to_numeric(df["dengue_cases"], errors="coerce")
df["district_key"] = df["district_key"].astype("string").str.strip().str.lower()

print("Shape:", df.shape)
print("Districts:", df["district_key"].nunique())
print("Date range:", df["week_start"].min(), "to", df["week_end"].max())
display(df[required_cols].head())

## 3. Data-quality checks

Do not silently remove suspicious records. Stop and investigate if required checks fail.

In [ ]:
df = df.sort_values(["district_key", "week_start"]).reset_index(drop=True)

quality = {
    "rows": len(df),
    "districts": df["district_key"].nunique(),
    "missing_district": df["district_key"].isna().sum(),
    "missing_week_start": df["week_start"].isna().sum(),
    "missing_week_end": df["week_end"].isna().sum(),
    "missing_dengue_cases": df["dengue_cases"].isna().sum(),
    "negative_dengue_cases": (df["dengue_cases"] < 0).sum(),
    "duplicate_district_week_start": df.duplicated(["district_key", "week_start"]).sum(),
    "week_end_before_week_start": (df["week_end"] < df["week_start"]).sum()
}
display(pd.DataFrame({"check": quality.keys(), "value": quality.values()}))

if any(quality[k] > 0 for k in [
    "missing_district", "missing_week_start", "missing_week_end",
    "missing_dengue_cases", "negative_dengue_cases",
    "duplicate_district_week_start", "week_end_before_week_start"
]):
    raise ValueError("Data-quality issue found. Review before continuing.")

print("Required data-quality checks passed.")

## 4. Inspect reporting-period lengths

The source includes 6-, 7-, and 8-day reporting periods. We calculate a seven-day-equivalent rate for comparability. The original `dengue_cases` column remains unchanged.

In [ ]:
df["period_days"] = (df["week_end"] - df["week_start"]).dt.days + 1

display(
    df["period_days"].value_counts().sort_index()
      .rename_axis("period_days").reset_index(name="records")
)

if (df["period_days"] <= 0).any():
    raise ValueError("Non-positive reporting period found.")

df["cases_per_7_days"] = df["dengue_cases"] / df["period_days"] * 7

display(df[[
    "district_key", "week_start", "week_end",
    "period_days", "dengue_cases", "cases_per_7_days"
]].head())

## 5. Explore district-level dengue patterns

Review historical peaks; do not remove them merely because they are extreme.

In [ ]:
display(
    df.groupby("district_key")["dengue_cases"]
      .agg(["count", "mean", "median", "std", "max"])
      .sort_values("mean", ascending=False)
)

selected_districts = ["colombo", "gampaha", "kandy", "jaffna"]
fig, axes = plt.subplots(len(selected_districts), 1, figsize=(14, 11), sharex=True)

for ax, district in zip(axes, selected_districts):
    part = df[df["district_key"] == district]
    ax.plot(part["week_start"], part["dengue_cases"], linewidth=0.8)
    ax.set_title(district.title())
    ax.set_ylabel("Cases")
    ax.grid(alpha=0.25)

axes[-1].set_xlabel("Reporting period start")
plt.tight_layout()
plt.show()

## 6. Calculate a district-specific historical baseline

For each district, the baseline uses the **previous 52 reporting periods**, excluding the current observation with `shift(1)`. At least 26 previous observations are required.

We use the median and median absolute deviation (MAD), which are less sensitive to extreme values than the mean and standard deviation.

This is a trailing baseline, not a seasonally matched baseline. Seasonal effects may influence the results and must be evaluated later.

In [ ]:
WINDOW_PERIODS = 52
MIN_BASELINE_PERIODS = 26

df["baseline_median_52"] = (
    df.groupby("district_key")["cases_per_7_days"]
      .transform(lambda s: s.shift(1).rolling(
          WINDOW_PERIODS, min_periods=MIN_BASELINE_PERIODS
      ).median())
)

def previous_rolling_mad(series):
    previous = series.shift(1)

    def calculate_mad(values):
        median = np.median(values)
        return np.median(np.abs(values - median))

    return previous.rolling(
        WINDOW_PERIODS, min_periods=MIN_BASELINE_PERIODS
    ).apply(calculate_mad, raw=True)

df["baseline_mad_52"] = (
    df.groupby("district_key")["cases_per_7_days"]
      .transform(previous_rolling_mad)
)

display(df.groupby("district_key").tail(2)[[
    "district_key", "week_start", "cases_per_7_days",
    "baseline_median_52", "baseline_mad_52"
]])

## 7. Calculate robust anomaly scores

\[
\text{Robust score} =
\frac{x - \text{historical median}}
{1.4826 \times \text{historical MAD}}
\]

If historical MAD is zero, the score is left missing. We do not divide by zero or manufacture an anomaly score.

In [ ]:
MAD_SCALE = 1.4826
df["robust_anomaly_score"] = np.nan

valid_mad = (
    df["baseline_median_52"].notna()
    & df["baseline_mad_52"].notna()
    & (df["baseline_mad_52"] > 0)
)

df.loc[valid_mad, "robust_anomaly_score"] = (
    df.loc[valid_mad, "cases_per_7_days"]
    - df.loc[valid_mad, "baseline_median_52"]
) / (MAD_SCALE * df.loc[valid_mad, "baseline_mad_52"])

df["zero_mad_baseline"] = (
    df["baseline_mad_52"].notna()
    & (df["baseline_mad_52"] == 0)
)

display(df[[
    "district_key", "week_start", "cases_per_7_days",
    "baseline_median_52", "baseline_mad_52",
    "robust_anomaly_score", "zero_mad_baseline"
]].tail(10))

## 8. Flag candidate high anomalies

Use a provisional robust-score threshold of **3.5 or greater** for this exploratory screen. It is not an official dengue outbreak definition and has not yet been validated for Sri Lanka.

Only observations with a valid, non-zero-MAD baseline can be flagged.

In [ ]:
PROVISIONAL_HIGH_ANOMALY_THRESHOLD = 3.5

df["candidate_high_anomaly"] = (
    df["robust_anomaly_score"] >= PROVISIONAL_HIGH_ANOMALY_THRESHOLD
)

df["anomaly_status"] = "not_flagged"
df.loc[
    df["robust_anomaly_score"].isna(),
    "anomaly_status"
] = "insufficient_or_zero_mad_baseline"
df.loc[
    df["candidate_high_anomaly"],
    "anomaly_status"
] = "candidate_high_anomaly"

display(
    df["anomaly_status"].value_counts()
      .rename_axis("status").reset_index(name="records")
)
display(
    df[df["candidate_high_anomaly"]]
      .groupby("district_key").size()
      .sort_values(ascending=False)
      .rename("candidate_count").reset_index()
)

## 9. Review candidate high anomalies

Inspect original case counts, seven-day-equivalent rates, historical baselines, and scores. Do not call these confirmed outbreaks without independent epidemiological verification.

In [ ]:
candidate_cols = [
    "district_key", "week_start", "week_end", "period_days",
    "dengue_cases", "cases_per_7_days", "baseline_median_52",
    "baseline_mad_52", "robust_anomaly_score", "anomaly_status"
]

candidate_anomalies = (
    df.loc[df["candidate_high_anomaly"], candidate_cols]
      .sort_values("robust_anomaly_score", ascending=False)
      .reset_index(drop=True)
)

display(candidate_anomalies.head(30))
print("Total candidate high anomalies:", len(candidate_anomalies))

## 10. Visualize one district's baseline and candidate anomalies

Change `PLOT_DISTRICT` after reviewing the candidate table.

In [ ]:
PLOT_DISTRICT = "colombo"

plot_data = df[df["district_key"] == PLOT_DISTRICT.lower()].copy()
if plot_data.empty:
    raise ValueError(f"No observations found for {PLOT_DISTRICT}")

fig, ax = plt.subplots(figsize=(15, 6))
ax.plot(
    plot_data["week_start"], plot_data["cases_per_7_days"],
    label="Observed cases per 7 days", linewidth=0.9
)
ax.plot(
    plot_data["week_start"], plot_data["baseline_median_52"],
    label="Previous 52-period rolling median", linewidth=1.2
)

points = plot_data[plot_data["candidate_high_anomaly"]]
ax.scatter(
    points["week_start"], points["cases_per_7_days"],
    label="Candidate high anomaly", s=35
)

ax.set_title(f"Dengue anomaly screening — {PLOT_DISTRICT.title()}")
ax.set_xlabel("Reporting period start")
ax.set_ylabel("Seven-day-equivalent dengue cases")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 11. Export scored observations, candidate anomalies, and method manifest

In [ ]:
scored_path = OUTPUT_DIR / "dengue_anomaly_scored_weekly.csv"
candidate_path = OUTPUT_DIR / "candidate_high_dengue_anomalies.csv"
manifest_path = OUTPUT_DIR / "anomaly_detection_manifest.json"

df.to_csv(scored_path, index=False)
candidate_anomalies.to_csv(candidate_path, index=False)

manifest = {
    "project": "RiskWeave",
    "notebook": "07_dengue_anomaly_detection.ipynb",
    "input_file": str(DATA_PATH.relative_to(PROJECT_ROOT)),
    "method": "District-specific trailing rolling median and MAD",
    "signal": "Seven-day-equivalent dengue case count",
    "baseline_window_reporting_periods": WINDOW_PERIODS,
    "minimum_baseline_observations": MIN_BASELINE_PERIODS,
    "current_observation_excluded_from_baseline": True,
    "robust_score_formula": "(value - rolling_median) / (1.4826 * rolling_MAD)",
    "provisional_high_anomaly_threshold": PROVISIONAL_HIGH_ANOMALY_THRESHOLD,
    "zero_mad_policy": "Score left missing; not automatically flagged",
    "interpretation": "Candidate statistical anomalies only; not confirmed outbreaks",
    "rows_processed": int(len(df)),
    "districts": int(df["district_key"].nunique()),
    "candidate_high_anomalies": int(df["candidate_high_anomaly"].sum())
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4)

print("Saved:", scored_path)
print("Saved:", candidate_path)
print("Saved:", manifest_path)
print("Candidate high anomalies:", manifest["candidate_high_anomalies"])

## 12. Limitations and next validation tasks

Before operational use or describing flags as outbreaks, RiskWeave needs further validation:

1. **Seasonality:** Compare this trailing baseline with a seasonally matched baseline, such as comparable epidemiological weeks in previous years.
2. **Threshold validation:** Test alternative thresholds and assess sensitivity and false-positive behaviour.
3. **Historical event review:** Compare candidate dates with official epidemiological reports and documented outbreak periods, where available.
4. **Surveillance changes:** Check whether reporting completeness, surveillance changes, or district definitions explain unusual values.
5. **District interpretation:** Review district-level results rather than applying one national case-count threshold.
6. **Dashboard language:** Until validated against authoritative outbreak criteria, use “candidate anomaly” or “unusual increase,” not “confirmed outbreak.”

### Completion checklist
- [ ] Data-quality checks reviewed
- [ ] Reporting-period normalization reviewed
- [ ] Historical baseline and robust scores calculated
- [ ] Candidate anomalies inspected
- [ ] District plots reviewed
- [ ] CSV outputs and manifest saved
- [ ] Candidate dates compared with official epidemiological evidence